In [1]:
import pandas as pd
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from pathlib import Path

Path("models").mkdir(exist_ok=True)
Path("outputs").mkdir(exist_ok=True)
Path("figures").mkdir(exist_ok=True)

df = pd.read_parquet("data/news_clean_ai.parquet")

docs = df["article_text"].tolist()

In [2]:
df_sample = df.sample(n=min(30000, len(df)), random_state=42)
docs = df_sample["article_text"].tolist()

In [3]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

topic_model = BERTopic(
    embedding_model=embedding_model,
    min_topic_size=100,
    verbose=True
)

topics, probs = topic_model.fit_transform(docs)

df_sample["topic"] = topics
df_sample["topic_prob"] = probs

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-05-19 15:51:37,570 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/938 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
topic_info = topic_model.get_topic_info()
topic_info.head(20)

In [ ]:
topic_info.to_csv("outputs/topic_info.csv", index=False)
df_sample.to_parquet("data/news_with_topics.parquet", index=False)
topic_model.save("models/bertopic_ai_news")

In [ ]:
topic_keywords = []

for topic_id in topic_info["Topic"].unique():
    if topic_id == -1:
        continue
    
    words = topic_model.get_topic(topic_id)
    if words:
        keywords = ", ".join([w for w, score in words[:10]])
        topic_keywords.append({
            "topic": topic_id,
            "keywords": keywords
        })

topic_keywords_df = pd.DataFrame(topic_keywords)
topic_keywords_df.to_csv("outputs/topic_keywords.csv", index=False)
topic_keywords_df.head(20)

In [ ]:
topic_labels = {
    0: "Generative AI and LLM Adoption",
    1: "Healthcare AI and Diagnostics",
    2: "AI in Finance and Risk Analytics",
    3: "Robotics and Manufacturing Automation",
    4: "Legal and Professional Services Automation",
    5: "AI Chips and Cloud Infrastructure",
}

In [ ]:
df_sample["topic_label"] = df_sample["topic"].map(topic_labels).fillna("Other / Mixed Topic")

df_sample.to_parquet("data/news_with_topics_labeled.parquet", index=False)

In [ ]:
import matplotlib.pyplot as plt

top_topics = (
    df_sample[df_sample["topic"] != -1]
    ["topic_label"]
    .value_counts()
    .head(12)
)

plt.figure(figsize=(10,6))
top_topics.sort_values().plot(kind="barh")
plt.xlabel("Number of Articles")
plt.title("Most Common AI Impact Topics")
plt.tight_layout()
plt.savefig("figures/topic_distribution.png", dpi=300)
plt.show()